# Несколько попыток захода

Тематический этап подготовки Европы 2019–2023. Выполняйте ячейки сверху вниз в окружении проекта. Каждый ноутбук соответствует одному шагу; диагностика и применение согласованного изменения разделены. Входы, выходы и порядок описаны в [карте подготовки](README.md). Сохраненные результаты перенесены из выполненного общего ноутбука; сам перенос данных не преобразует.

## Шаг 10 – записи с большим числом попыток

Проверяем весь файл, затем отдельно разбираем n_attempts > 2. Этот порог служит диагностике, а не авиационной норме. [Описание набора](https://zenodo.org/records/10210178) определяет число обнаруженных подходов; GoA отмечает наличие хотя бы одного ухода. Число n_attempts - 1 не принимаем за подтвержденное число эксплуатационных уходов. ILS содержит обозначения ВПП, а не доказательство фактического применения системы ILS.

Сохраняем времена, интервалы и исходные строки. Длительность между первой и последней попытками не является полной длительностью полета. Проверяем контрольные суммы исходника и рабочей копии.

In [1]:
# Готовим разбор нескольких попыток и интервальных характеристик
from pathlib import Path
import json
import runpy
import pandas as pd
# Находим папку Европы из любой вложенной папки ноутбуков
region = next(p / 'europe' for p in [Path.cwd(), *Path.cwd().parents]
              if (p / 'europe/data/source/osn23_landings_merged.parquet.gzip').exists())
# Скрипт проверяет все строки, сохраняет группу n_attempts > 2 и контрольные суммы
attempt_module = runpy.run_path(str(region / 'scripts/diagnose_attempts.py'))
attempt_counts, attempt_airports, high_attempts, attempt_intervals = attempt_module['main']()
attempt_journal = json.loads((region / 'reports/preparation/09_attempts_diagnostics.json').read_text(encoding='utf8'))
# Сравниваем число попыток и показываем структурные противоречия
display(attempt_counts)
display(pd.Series(attempt_journal['structural_flags'], name='Число противоречий'))

Проверено строк: 1000000


Проверено строк: 2000000


Проверено строк: 3000000


Проверено строк: 4000000


Проверено строк: 5000000


Проверено строк: 6000000


Проверено строк: 7000000


{
  "input": "europe\\data\\working\\03_c40_diagnostic_flags.parquet",
  "input_sha256": "ddf11438bab44137242ffcf4da928418e116efd46defce58dc8a388b67942b6d",
  "source_sha256": "1160bdbea6fa93249a00768452be64047c6d2461bb0843d5ecfc52e6942c7f02",
  "input_unchanged": true,
  "source_unchanged": true,
  "rows_checked": 7984254,
  "rows_changed": 0,
  "rows_deleted": 0,
  "diagnostic_definition": "n_attempts > 2; not an aviation limit",
  "high_attempt_rows": 1052,
  "two_attempt_rows": 25201,
  "go_arounds": 26253,
  "high_attempt_percent_of_go_around_rows": 4.007161086352036,
  "max_attempts": 33,
  "high_attempt_airports": 20,
  "high_attempt_aircraft_addresses": 361,
  "structural_flags": {
    "attempt_count_nonpositive": 0,
    "attempt_length_mismatch": 0,
    "ILS_length_mismatch": 0,
    "GoA_differs_from_more_than_one_attempt": 0,
    "null_attempt_element": 0,
    "descending_attempt_times": 0,
    "repeated_adjacent_attempt_time": 0,
    "last_attempt_after_landing": 0
  },
  "h

,n_attempts,rows,aircraft_addresses,airports,span_min_minutes,span_median_minutes,span_max_minutes,rows_same_runway,c40_missing_rows
0,2,25201,7575,20,0.200000,13.366667,171.383333,21681,6946
1,3,421,295,20,9.316667,30.683333,241.933333,268,277
2,4,104,53,18,11.233333,52.541667,211.266667,64,99
3,5,81,38,17,16.916667,62.050000,270.516667,40,80
4,6,69,29,16,27.150000,86.783333,233.066667,36,69
5,7,38,15,11,38.400000,104.866667,195.800000,18,38
6,8,21,11,12,45.583333,113.483333,250.533333,7,21
7,9,24,16,11,50.700000,104.583333,259.133333,6,23
8,10,19,11,10,61.983333,119.683333,230.950000,9,19
9,11,17,10,9,98.550000,146.450000,230.300000,4,17


attempt_count_nonpositive                 0
attempt_length_mismatch                   0
ILS_length_mismatch                       0
GoA_differs_from_more_than_one_attempt    0
null_attempt_element                      0
descending_attempt_times                  0
repeated_adjacent_attempt_time            0
last_attempt_after_landing                0
Name: Число противоречий, dtype: int64

In [2]:
# Показываем распределение по аэропортам и пять максимальных чисел попыток
display(attempt_airports)
with pd.option_context('display.max_columns', None):
    display(high_attempts.nlargest(5, 'n_attempts'))
# Различаем медиану отдельных промежутков и медиану окна по наблюдениям
print('Медиана интервала, минуты:', attempt_journal['high_attempt_gap_minutes']['median'])
print('Медиана окна первой–последней попытки, минуты:', attempt_journal['high_attempt_span_minutes']['median'])

,airport,rows,aircraft_addresses,max_attempts,median_span_minutes
0,EDDB,83,32,20,72.250000
1,EDDF,101,30,25,172.666667
2,EDDM,61,12,23,155.666667
3,EGCC,14,7,13,97.758333
4,EGLL,18,17,3,32.141667
5,EGSS,33,16,12,33.616667
6,EHAM,80,38,28,26.008333
7,EIDW,137,19,15,58.233333
8,EKCH,25,7,28,116.666667
9,EPWA,64,19,24,67.591667


,source_row_position,flight_id,icao24,callsign,registration,airport,landing_time,year,n_attempts,GoA,typecode,icaoaircrafttype,market_segment,first_attempt_time,last_attempt_time,first_to_last_attempt_minutes,first_attempt_to_landing_minutes,last_attempt_to_landing_seconds,min_attempt_gap_minutes,median_attempt_gap_minutes,max_attempt_gap_minutes,unique_runway_designations,runway_switches,c40_time_missing,c40_after_first_attempt,c40_after_landing
25877,7875471,FCK1EL_3815,3cce70,FCK1EL,DCFME,LOWW,2023-05-12 02:35:51+00:00,2023,33,True,B350,L2T,NaN,2023-05-11 22:19:01+00:00,2023-05-12 02:35:47+00:00,256.766667,256.833333,4.0,4.966667,6.100000,34.333333,1,0,True,False,False
10468,3378043,FCK511_5650,3cce71,FCK511,NaN,LOWW,2020-09-23 00:54:10+00:00,2020,31,True,NaN,NaN,NaN,2020-09-22 21:04:27+00:00,2020-09-23 00:53:58+00:00,229.516667,229.716667,12.0,4.816667,5.458333,27.283333,2,2,True,False,False
10478,3381024,FCK511_5651,3cce71,FCK511,NaN,LOWW,2020-09-23 23:59:58+00:00,2020,30,True,NaN,NaN,NaN,2020-09-23 20:40:40+00:00,2020-09-23 23:59:36+00:00,198.933333,199.300000,22.0,4.733333,5.883333,14.683333,2,3,True,False,False
12081,3828528,FCK511_4865,3cce71,FCK511,NaN,LOWW,2021-04-12 15:26:51+00:00,2021,30,True,NaN,NaN,Other,2021-04-12 10:44:18+00:00,2021-04-12 15:26:49+00:00,282.516667,282.550000,2.0,5.533333,6.483333,69.533333,2,1,True,False,False
18736,5735838,FCK311_8302,3cce70,FCK311,DCFME,LOWW,2022-05-17 01:49:50+00:00,2022,29,True,B350,L2T,NaN,2022-05-16 22:01:31+00:00,2022-05-17 01:49:24+00:00,227.883333,228.316667,26.0,3.116667,5.933333,23.150000,2,1,True,False,False


Медиана интервала, минуты: 9.016666666666667
Медиана окна первой–последней попытки, минуты: 60.875


In [3]:
# Короткие промежутки и полные последовательности требуют траекторного разбора
with pd.option_context('display.max_columns', None):
    display(high_attempts.nsmallest(5, 'min_attempt_gap_minutes'))
    display(pd.read_csv(region / 'reports/preparation/09_high_attempt_examples.csv'))
# Проверяем пересечение с проблемами C40 и сохранность исходных данных
print('C40 после первой попытки:', attempt_journal['high_attempt_c40_after_first_rows'])
print('Исходник и рабочая копия неизменны:', attempt_journal['source_unchanged'], attempt_journal['input_unchanged'])
print('Изменено / удалено строк:', attempt_journal['rows_changed'], attempt_journal['rows_deleted'])

,source_row_position,flight_id,icao24,callsign,registration,airport,landing_time,year,n_attempts,GoA,typecode,icaoaircrafttype,market_segment,first_attempt_time,last_attempt_time,first_to_last_attempt_minutes,first_attempt_to_landing_minutes,last_attempt_to_landing_seconds,min_attempt_gap_minutes,median_attempt_gap_minutes,max_attempt_gap_minutes,unique_runway_designations,runway_switches,c40_time_missing,c40_after_first_attempt,c40_after_landing
12577,3954070,ZXP24_100524,485b2f,ZXP24,PHPXX,EHAM,2021-06-04 06:48:05+00:00,2021,3,True,NaN,NaN,NaN,2021-06-04 06:22:36+00:00,2021-06-04 06:47:40+00:00,25.066667,25.483333,25.0,0.650000,12.533333,24.416667,2,2,True,False,False
16714,5177528,ZXP24_145897,485b2f,ZXP24,PHPXX,EHAM,2022-02-10 19:29:20+00:00,2022,3,True,NaN,NaN,NaN,2022-02-10 17:57:53+00:00,2022-02-10 19:28:20+00:00,90.450000,91.450000,60.0,0.650000,45.225000,89.800000,2,2,True,False,False
23316,7157626,PGT1SC_148296,4bb84b,PGT1SC,TCNBK,EDDF,2022-12-12 09:31:11+00:00,2022,3,True,A20N,L2J,Lowcost,2022-12-12 09:17:54+00:00,2022-12-12 09:30:54+00:00,13.000000,13.283333,17.0,1.466667,6.500000,11.533333,1,0,False,False,False
20083,6131764,DLH419_20578,3c4b24,DLH419,DABYD,EDDF,2022-07-12 06:44:27+00:00,2022,3,True,B748,L4J,Mainline,2022-07-12 06:30:54+00:00,2022-07-12 06:44:27+00:00,13.550000,13.550000,0.0,2.433333,6.775000,11.116667,1,0,False,False,False
10306,3313179,PROVA20_4454,33fd75,PROVA20,NaN,LIMC,2020-09-03 15:40:46+00:00,2020,4,True,NaN,NaN,NaN,2020-09-03 15:15:51+00:00,2020-09-03 15:27:05+00:00,11.233333,24.916667,821.0,2.550000,2.600000,6.083333,1,0,True,False,False


,source_row_position,flight_id,icao24,airport,n_attempts,attempt_number_from,attempt_number_to,time_from,time_to,runway_from,runway_to,gap_minutes,same_runway
0,3378043,FCK511_5650,3cce71,LOWW,31,1,2,2020-09-22 21:04:27+00:00,2020-09-22 21:17:32+00:00,16,16,13.083333,True
1,3378043,FCK511_5650,3cce71,LOWW,31,2,3,2020-09-22 21:17:32+00:00,2020-09-22 21:24:25+00:00,16,16,6.883333,True
2,3378043,FCK511_5650,3cce71,LOWW,31,3,4,2020-09-22 21:24:25+00:00,2020-09-22 21:30:28+00:00,16,16,6.050000,True
3,3378043,FCK511_5650,3cce71,LOWW,31,4,5,2020-09-22 21:30:28+00:00,2020-09-22 21:36:07+00:00,16,16,5.650000,True
4,3378043,FCK511_5650,3cce71,LOWW,31,5,6,2020-09-22 21:36:07+00:00,2020-09-22 21:41:37+00:00,16,16,5.500000,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...
92,7875471,FCK1EL_3815,3cce70,LOWW,33,28,29,2023-05-12 01:50:50+00:00,2023-05-12 01:56:56+00:00,11,11,6.100000,True
93,7875471,FCK1EL_3815,3cce70,LOWW,33,29,30,2023-05-12 01:56:56+00:00,2023-05-12 02:09:33+00:00,11,11,12.616667,True
94,7875471,FCK1EL_3815,3cce70,LOWW,33,30,31,2023-05-12 02:09:33+00:00,2023-05-12 02:15:53+00:00,11,11,6.333333,True
95,7875471,FCK1EL_3815,3cce70,LOWW,33,31,32,2023-05-12 02:15:53+00:00,2023-05-12 02:21:57+00:00,11,11,6.066667,True


C40 после первой попытки: 3
Исходник и рабочая копия неизменны: True True
Изменено / удалено строк: 0 0


## Итог шага 10

Выделено 1 052 записи (4,01% строк GoA=True), максимум 33 попытки. Противоречий длины списков, порядка времен и связи GoA с числом попыток не обнаружено. Это структурная согласованность, а не подтверждение каждой метки по траектории.

У записи с 33 попытками указан DCFME. [Оператор FCS](https://fcs.aero/en/about-fcs/) описывает D-CFME как самолет летных проверок. Эта информация поддерживает гипотезу о специальном назначении, но не устанавливает задачу конкретного полета 2023 года. Другие записи автоматически учебными или проверочными не считаем. Текущая документация traffic не доказывает точную историческую конфигурацию алгоритма.

Сохраняем все записи. Число попыток, итоговые времена и последовательность ВПП известны после события и требуют исключения из предсобытийных признаков. Подробности и отложенные решения: [разбор](../../reports/preparation/09_attempts_review.md). Следующий шаг – свести результаты диагностики в журнал правил и определить состав наблюдения для моделирования до очистки аэропортов и строк.